In [ ]:
# 03_baseline_xgboost.ipynb

# 1. Imports
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.src_bak import data_loader, features, volatility, regimes, models_xgb

# 2. The Complete Modular Pipeline Execution
raw_df = data_loader.fetch_raw_data()
stat_df = features.engineer_stationary_features(raw_df)

# Generate ML-Safe GARCH
garch_df = volatility.generate_expanding_garch(stat_df, window_size=252)

# Generate Targets
regime_df = regimes.generate_smoothed_targets(
    garch_df, lower_quant=0.85, upper_quant=0.95, floor=0.0
)

# Generate Lags for XGBoost
cols_to_lag = [
    'Log_Return', 'Sq_Log_Return', 'Vol_GARCH', 'Vol_EGARCH', 
    'VIX_Change', 'Oil_Change', 'CPI_MoM', 'FedFunds_Diff', 'Term_Spread_Diff'
]
final_df = features.create_lags(regime_df, feature_cols=cols_to_lag)

# 3. Train and Evaluate
model, feature_names, X_test, y_test = models_xgb.train_xgboost_baseline(final_df)

# 4. Feature Importance
models_xgb.plot_feature_importance(model, feature_names)

In [ ]:
# In 03_baseline_xgboost.ipynb
from src.src_bak import data_loader, features, volatility, regimes, models_xgb

# 1. Core Pipeline
raw_df = data_loader.fetch_raw_data()
stat_df = features.engineer_stationary_features(raw_df)
garch_df = volatility.generate_expanding_garch(stat_df, window_size=252)
regime_df = regimes.generate_smoothed_targets(garch_df, lower_quant=0.85, upper_quant=0.95, floor=0.0)

# 2. Lag Engineering
cols_to_lag = ['Log_Return', 'Sq_Log_Return', 'Vol_GARCH', 'Vol_EGARCH', 
               'VIX_Change', 'Oil_Change', 'CPI_MoM', 'FedFunds_Diff', 'Term_Spread_Diff']
final_df = features.create_lags(regime_df, feature_cols=cols_to_lag)

# 3. Run the Walk-Forward Validation (4 splits = 20% chunks)
final_model, feature_names = models_xgb.train_xgb_walk_forward(final_df, n_splits=4)

In [ ]:
# 4. Look at feature importance from the most mature model fold
models_xgb.plot_feature_importance(final_model, feature_names)

In [ ]:
# 03_baseline_xgboost.ipynb
from src.src_bak import data_loader, features, volatility, regimes, models_xgb
import pandas as pd

# 1. Standard Data Ingest & Math
raw_df = data_loader.fetch_raw_data()
stat_df = features.engineer_stationary_features(raw_df)
garch_df = volatility.generate_expanding_garch(stat_df)

# 2. Generate the 4-State HMM
print("\n--- Generating HMM Targets ---")
hmm_4_df = regimes.generate_hmm_targets(garch_df, feature_cols=['Log_Return', 'Vol_EGARCH'], n_components=4)

# 3. Safe Merge & Mapping to 3 ML Classes
master_df = garch_df.copy()
master_df['Target_HMM'] = hmm_4_df['Target_HMM']
master_df = master_df.dropna(subset=['Target_HMM'])

# Collapse the two lowest volatility states into Class 0 (Calm)
mapping_dict = {
    0: 0,  # Dead Calm -> Calm
    1: 0,  # Normal -> Calm
    2: 1,  # Elevated
    3: 2   # Shock
}
master_df['Target_HMM_ML'] = master_df['Target_HMM'].map(mapping_dict).astype(int)
master_df = master_df.drop('Target_HMM', axis=1)

# 4. Lag Engineering for XGBoost
cols_to_lag = [
    'Log_Return', 'Sq_Log_Return', 'Vol_GARCH', 'Vol_EGARCH', 
    'VIX_Change', 'Oil_Change', 'CPI_MoM', 'FedFunds_Diff', 'Term_Spread_Diff'
]
final_df = features.create_lags(master_df, feature_cols=cols_to_lag)

# 5. Walk-Forward Validation
# We pass our newly mapped HMM column as the target
final_model, feature_names = models_xgb.train_xgb_walk_forward(
    df=final_df, 
    target_col='Target_HMM_ML', 
    n_splits=4
)

In [ ]:
# 6. Plot the Final Feature Importance
models_xgb.plot_feature_importance(final_model, feature_names)